In [28]:
from langgraph.graph import StateGraph,START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv
load_dotenv()
import os

In [29]:
llm = ChatOpenAI(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv('GROQ_API_KEY'),
    base_url="https://api.groq.com/openai/v1"
)

In [30]:
# crate a state 
class LLMState(TypedDict):
    question: str
    answer: str
    description: str

In [31]:
def llm_qa(state: LLMState) -> LLMState:

    # extract the question from the state
    question = state['question']

    # create a prompt for the LLM
    prompt = f"Answer the following question: {question}"

    # ask the LLM to answer the question
    answer = llm.invoke(prompt).content

    # update the state with the answer
    state['answer'] = answer

    return state
llm_qa({'question': 'What is the capital of France?'})

{'question': 'What is the capital of France?',
 'answer': 'The capital of France is Paris.'}

In [32]:
def describe_answer(state: LLMState) -> LLMState:

    # extract the answer from the state
    answer = state['answer']

    # create a prompt for the LLM to describe the answer
    prompt = f"Describe the following answer in detail: {answer}"

    # ask the LLM to describe the answer
    description = llm.invoke(prompt).content

    # update the state with the description
    state['description'] = description

    return state

In [34]:
# create a state graph
graph = StateGraph(LLMState)

# add nodes to the graph
graph.add_node('llm_qa',llm_qa)
graph.add_node('describe_answer', describe_answer)

# add edges to the graph
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', 'describe_answer')
graph.add_edge('describe_answer', END)

workflow = graph.compile()
workflow.invoke({'question': 'What is the capital of France?'})

{'question': 'What is the capital of France?',
 'answer': 'The capital of France is Paris.',
 'description': 'The statement "The capital of France is Paris" is a straightforward and factual assertion that provides information about the capital city of a specific country, namely France.\n\n**Breaking down the statement:**\n\n1. **"The capital"**: This phrase refers to the primary city or administrative center of a country, where the government and its main institutions are typically located.\n2. **"of France"**: This prepositional phrase indicates the country to which the capital city belongs. France is a sovereign nation located in Western Europe, known for its rich history, culture, and iconic landmarks.\n3. **"is Paris"**: This verb phrase links the subject (the capital) to the object (Paris), stating that Paris is the capital city of France. Paris is a global city, famous for its stunning architecture, art museums, fashion, and culinary scene.\n\n**Significance of Paris as the capit

In [36]:
# execute the workflow
initial_state = {'question': 'What is the capital of France?'}
final_state = workflow.invoke(initial_state)
print(final_state['description'])  # This will print the answer to the question

The statement "The capital of France is Paris" is a concise and straightforward answer that provides a fundamental piece of information about the country of France. Here's a detailed breakdown of this response:

1. **Subject**: The subject of the sentence is "The capital of France", which refers to the administrative and governmental hub of the country. In this context, the term "capital" is used to denote the most important city in France, where the central government, national institutions, and key decision-making bodies are located.

2. **Predicate**: The predicate of the sentence is "is Paris", which states that the city of Paris holds the position of capital. This predicate provides the essential information that the question or inquiry is seeking, i.e., the name of the capital city of France.

3. **Geographical and Cultural Context**: France is a country located in Western Europe, known for its rich history, culture, art, and architecture. The city of Paris, being the capital, is

In [37]:
# we can simply displa answer from simple llm
llm.invoke("What is the capital of France?").content

'The capital of France is Paris.'